Serve apenas para diagnóstico da silver de chamados hora. Apenas para verificar se os resultados estão batendo.

In [0]:
from pyspark.sql import functions as F, Window

catalog = "rocket_lab"
bronze_schema = f"{catalog}.vcredit_bronze"
silver_schema = f"{catalog}.vcredit_silver"
tabela_destino = f"{silver_schema}.chamados_hora"

HORAS = ["hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento"]

In [0]:
ch = spark.table(f"{silver_schema}.chamados").select("id_chamado", "canal", "tempo_espera")

hr = (spark.table(tabela_destino)
      .select("id_chamado", "hora_abertura_chamado", "hora_inicio_atendimento"))

x = (ch.join(hr, "id_chamado", "left")
       .withColumn("espera_calc",
                   F.unix_timestamp("hora_inicio_atendimento") - F.unix_timestamp("hora_abertura_chamado")))

# por canal: quantos têm espera nula, quantos teriam espera 0 pelos horários, quantos não têm horário
display(x.groupBy("canal").agg(
            F.count("*").alias("chamados"),
            F.sum(F.col("tempo_espera").isNull().cast("int")).alias("espera_nula"),
            F.sum((F.col("espera_calc") == 0).cast("int")).alias("espera_calc_zero"),
            F.sum(F.col("hora_abertura_chamado").isNull().cast("int")).alias("sem_horario"))
        .orderBy(F.desc("chamados")))

# onde a espera existe nos dois lados, ela bate?
comp = x.filter(F.col("tempo_espera").isNotNull() & F.col("espera_calc").isNotNull())
display(comp.select(F.count("*").alias("comparaveis"),
                    F.sum((F.col("tempo_espera") == F.col("espera_calc")).cast("int")).alias("batem")))

In [0]:
bronze_cli = (spark.table(f"{bronze_schema}.clientes")
              .select(F.trim("id_cliente").alias("id_cliente")).distinct()
              .withColumn("existe_na_bronze", F.lit(True)))

nao_encontrados = (spark.table(tabela_destino).filter(~F.col("cliente_valido"))
                   .join(bronze_cli, "id_cliente", "left")
                   .withColumn("existe_na_bronze", F.coalesce("existe_na_bronze", F.lit(False))))

display(nao_encontrados.groupBy("existe_na_bronze").count())

In [0]:
# 1) os 3.414 sem início: abandono ou início recuperável?
si = spark.table(tabela_destino).filter("sem_inicio_atendimento")
chs = spark.table(f"{silver_schema}.chamados").select(
    "id_chamado", "canal", "resolvido", "tempo_espera", "tempo_atendimento")
display(si.join(chs, "id_chamado", "left")
          .groupBy("canal", "resolvido")
          .agg(F.count("*").alias("chamados"),
               F.sum(F.col("tempo_espera").isNotNull().cast("int")).alias("com_tempo_espera"),
               F.sum(F.col("tempo_atendimento").isNotNull().cast("int")).alias("com_tempo_atendimento"))
          .orderBy(F.desc("chamados")))

# 2) tamanho da divergência entre tempo_espera e os horários
d = comp.withColumn("dif", F.col("tempo_espera") - F.col("espera_calc")).filter("dif != 0")
display(d.agg(F.count("*").alias("divergentes"),
              F.min("dif").alias("dif_min"), F.max("dif").alias("dif_max"),
              F.expr("percentile_approx(dif, array(0.1, 0.5, 0.9))").alias("p10_p50_p90")))

# 3) as linhas repetidas do chamados na bronze são cópias idênticas?
b = spark.table(f"{bronze_schema}.chamados")
b = b.toDF(*[c.lower() for c in b.columns]).withColumn(
    "id_chamado", F.expr("try_cast(trim(id_chamado) as bigint)"))
cols = [c for c in b.columns if c not in ("ingest_timestamp", "source_file")]
print("linhas:", b.count(),
      "| ids distintos:", b.select("id_chamado").distinct().count(),
      "| linhas distintas (todas as colunas):", b.select(*cols).distinct().count())

In [0]:
chs2 = spark.table(f"{silver_schema}.chamados").select("id_chamado", "tempo_atendimento")
fim_df = spark.table(tabela_destino).select("id_chamado", "hora_finalizacao_atendimento")

dd = (d.join(chs2, "id_chamado").join(fim_df, "id_chamado")
        .withColumn("atend_calc",
                    F.unix_timestamp("hora_finalizacao_atendimento") - F.unix_timestamp("hora_inicio_atendimento")))

display(dd.agg(F.count("*").alias("divergentes"),
               F.sum((F.col("tempo_atendimento") == F.col("atend_calc")).cast("int")).alias("atendimento_bate")))